**Role in the paper.** SI figure of the simulation domains (orography of the two nested ALARO1-SFX domains) with the nine cities. Reads `data/metadata/simulations/orog_euro{1,2}.nc` and saves `results/figures/Supplementary_Information/simulation_domain_with_cities.png`.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib as mpl
from rasterio import warp
from pathlib import Path
ROOT = Path.cwd().parents[1]  # run from scripts/plotting

In [ ]:
def load_orog(level, data_dir):
    filename = f"orog_euro{level}.nc"
    filepath = os.path.join(data_dir, filename)

    if not os.path.exists(filepath):
        raise FileNotFoundError(f"OROG file not found at {filepath}")

    ds = xr.load_dataset(filepath)

    ds_crs_dict = ds.crs.attrs
    print(f"Loaded OROG data for level {level} with CRS: {ds_crs_dict.keys()}")
    ds_crs = ccrs.LambertConformal(
        central_latitude=ds_crs_dict["latitude_of_projection_origin"],
        central_longitude=ds_crs_dict["longitude_of_central_meridian"],
        standard_parallels=(ds_crs_dict["standard_parallel"]),
        globe=ccrs.Globe(
            semimajor_axis=ds_crs_dict["earth_radius"],
            semiminor_axis=ds_crs_dict["earth_radius"],
        ),
    )
    return ds, ds_crs


def load_domain_shapes(shp_path):
    if not os.path.exists(shp_path):
        raise FileNotFoundError(f"Shapefile not found at {shp_path}")

    gdf = gpd.read_file(shp_path)
    return gdf


def add_base_features(ax):
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), facecolor="#cfe8f3", zorder=1)
    ax.add_feature(cfeature.LAKES.with_scale("50m"), facecolor="#cfe8f3", zorder=1)
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), linewidth=0.6, zorder=3)
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.4, zorder=3, color= 'gray', ls='--')


def add_gridlines(ax, labelsize=10, draw_right_labels=True, draw_top_labels=True):
    gl = ax.gridlines(
        draw_labels=True,
        dms=True,
        x_inline=False,
        y_inline=False,
        linewidth=0.6,
        color="gray",
        alpha=0.45,
        zorder=2,
    )
    gl.top_labels = draw_top_labels
    gl.bottom_labels = False
    gl.left_labels = True
    gl.right_labels = draw_right_labels
    gl.xlabel_style = {"size": labelsize, "color": "black"}
    gl.ylabel_style = {"size": labelsize, "color": "black"}
    return gl


def plot_domain_boundaries(ax, gdf):
    styles = [
        dict(idx=0, color="blue",   lw=2.4, ls="-",  label="ALARO-25km"),
        dict(idx=1, color="orange", lw=2.4, ls="--", label="ALARO-12km"),
        dict(idx=1, color="purple", lw=2.2, ls=":",  label="ALARO-12km-noSFX"),
        dict(idx=2, color="green",  lw=2.6, ls="-",  label="ALARO-4km"),
        dict(idx=3, color="red",    lw=2.6, ls="-",  label="ALARO-1.3km"),
    ]

    for s in styles:
        gdf.loc[[s["idx"]]].boundary.plot(
            ax=ax,
            transform=ccrs.PlateCarree(),
            color=s["color"],
            linewidth=s["lw"],
            linestyle=s["ls"],
            zorder=4,
            joinstyle="round",
            capstyle="round",
            antialiased=True,
        )
    return styles


def plot_zoom_boundaries(ax, gdf):
    styles = [
        dict(idx=2, color="green", lw=2.8, ls="-", label="ALARO-4km"),
        dict(idx=3, color="red",   lw=2.8, ls="-", label="ALARO-1.3km"),
    ]

    for s in styles:
        gdf.loc[[s["idx"]]].boundary.plot(
            ax=ax,
            transform=ccrs.PlateCarree(),
            color=s["color"],
            linewidth=s["lw"],
            linestyle=s["ls"],
            zorder=4,
            joinstyle="round",
            capstyle="round",
            antialiased=True,
        )
    return styles


def plot_nested_orography(ax, orog_list, orog_crs_list, cmap, norm):
    """
    Plot coarse-to-fine orography successively to highlight added detail
    in nested domains.
    """
    mp = None
    for orog, orog_crs in zip(orog_list, orog_crs_list):
        mp = orog.orog.plot(
            ax=ax,
            transform=orog_crs,
            cmap=cmap,
            norm=norm,
            add_colorbar=False,
            zorder=0,
            rasterized=True,
        )
    return mp


def add_direct_zoom_labels(ax):
    ax.text(
        0.8, 51.9, "4 km",
        color="green", fontsize=9,
        transform=ccrs.PlateCarree(), zorder=5
    )
    ax.text(
        2.2, 50.95, "1.3 km",
        color="red", fontsize=9,
        transform=ccrs.PlateCarree(), zorder=5
    )

def from_latlon_to_xy( lat, lon, proj):
    xy = rasterio.warp.transform(ccrs.PlateCarree(), proj, [lon], [lat])
    x, y = xy[0][0], xy[1][0]
    #print(f"Coordinates : x = {x:.2f} m, y = {y:.2f} m")
    return (x, y)


def plot_city_locations(ax, cities, latlon, proj):
    xy = [from_latlon_to_xy(lat, lon, proj) for lat, lon in latlon]
    x = [x[0] for x in xy]
    y = [x[1] for x in xy]
    ax.scatter(x,y, color='black', s=12, marker='o', zorder=10000)
    for i, txt in enumerate(cities):
        ax.annotate(txt, (x[i], y[i]), xytext=(5,1), textcoords='offset points', fontsize= 8 , color= 'black', weight='bold', zorder=10010)

def plot_simulation_domain_withcities(orog_list, orog_crs_list, cities, cities_latlon, output_path="./domain_cities"):
    proj = orog_crs_list[0]

    from matplotlib.gridspec import GridSpec


    fig = plt.figure(figsize=(5.8, 10.8))

    gs = GridSpec(
        3, 1,
        height_ratios=[1, 1, 0.05],
        figure=fig
    )

    ax1 = fig.add_subplot(gs[0], projection=proj)


    axs = [ax1]

    # Terrain colormap
    base_cmap = plt.get_cmap("terrain")
    colors = base_cmap(np.linspace(0.18, 0.95, 256))
    terrain_cmap = mpl.colors.LinearSegmentedColormap.from_list("terrain_trunc", colors)
    norm = mpl.colors.Normalize(vmin=0, vmax=3000)

    # -------------------------
    # Panel (a): Europe overview
    # -------------------------
    ax = axs[0]
    add_base_features(ax)
    mp = plot_nested_orography(ax, orog_list, orog_crs_list, terrain_cmap, norm)
    add_gridlines(ax, labelsize=9, draw_right_labels=False, draw_top_labels=True)
    
    plot_city_locations(ax, cities, cities_latlon, proj)



    ax.set_title(None)  # Remove default title
    fig.savefig(os.path.join(output_path, "simulation_domain_with_cities.png"), dpi=300, bbox_inches="tight")
    fig.savefig(os.path.join(output_path, "simulation_domain_with_cities.pdf"), dpi=300, bbox_inches="tight")



In [ ]:
figure_dir = str(ROOT / "results/figures/Supplementary_Information")

cities = ["AMS", "BCN", "BER", "BIR", "GHE", "LIS", "REN", "TUR", "ZUR"]
cities_latlons = [(52.36705057729636, 4.903261250676051), (41.38735433848073, 2.168390570228181), (52.520355269780985, 13.405708562830844), (52.482300311881886, -1.8904501089524361), (51.05454277446639, 3.7249005076743438), (38.72185501907889, -9.139708387802852), (48.11751888591575, -1.677997294481546), (60.45128573100419, 22.248242707978694), (47.37656400426996, 8.54144167156696)]
plot_simulation_domain_withcities(orog_list, orog_crs_list,cities, cities_latlons, output_path=figure_dir)

In [ ]:

data_dir = str(ROOT / "data/metadata/simulations")

shp_path = os.path.join(data_dir, "sara", "sara.shp")
#gdf = load_domain_shapes(shp_path)
gdf = 1

orog1 = xr.load_dataset(os.path.join(data_dir, "orog_euro1.nc"))
orog1_crs_dict = orog1.crs.attrs
orog1_crs = ccrs.LambertConformal(
        central_latitude=orog1_crs_dict["latitude_of_projection_origin"],
        central_longitude=orog1_crs_dict["longitude_of_central_meridian"],
        standard_parallels=([orog1_crs_dict["standard_parallel"]]),
        globe=ccrs.Globe(
            semimajor_axis=orog1_crs_dict["earth_radius"],
            semiminor_axis=orog1_crs_dict["earth_radius"],
        ),
    )

orog2 = xr.load_dataset(os.path.join(data_dir, "orog_euro2.nc"))
orog2_crs_dict = orog2.crs.attrs
orog2_crs = ccrs.LambertConformal(
        central_latitude=orog2_crs_dict["latitude_of_projection_origin"],
        central_longitude=orog2_crs_dict["longitude_of_central_meridian"],
        standard_parallels=(orog2_crs_dict["standard_parallel"]),
        globe=ccrs.Globe(
            semimajor_axis=orog2_crs_dict["earth_radius"],
            semiminor_axis=orog2_crs_dict["earth_radius"],
        ),
    )


orog_list = [orog1, orog2]
orog_crs_list = [orog1_crs, orog2_crs]

figure_dir = str(ROOT / "results/figures/Supplementary_Information")

plot_simulation_domains_two_panel(
    orog_list=orog_list,
    orog_crs_list=orog_crs_list,
    gdf=gdf,
    output_path=os.path.join(figure_dir, "simulation_domains"),
    horizontal=False,
)

In [ ]:
orog2 = xr.load_dataset(os.path.join(data_dir, "orog_euro2.nc"))
orog2_crs_dict = orog2.crs.attrs
orog2_crs = ccrs.LambertConformal(
        central_latitude=orog2_crs_dict["latitude_of_projection_origin"],
        central_longitude=orog2_crs_dict["longitude_of_central_meridian"],
        standard_parallels=(orog2_crs_dict["standard_parallel"]),
        globe=ccrs.Globe(
            semimajor_axis=orog2_crs_dict["earth_radius"],
            semiminor_axis=orog2_crs_dict["earth_radius"],
        ),
    )

In [ ]:
orog1_crs = ccrs.LambertConformal(
        central_latitude=orog1_crs_dict["latitude_of_projection_origin"],
        central_longitude=orog1_crs_dict["longitude_of_central_meridian"],
        standard_parallels=([orog1_crs_dict["standard_parallel"]]),
        globe=ccrs.Globe(
            semimajor_axis=orog1_crs_dict["earth_radius"],
            semiminor_axis=orog1_crs_dict["earth_radius"],
        ),
    )

In [ ]:
proj_dict = ds_RCM_urb.crs.attrs
plot_proj = ccrs.LambertConformal(central_latitude=proj_dict["latitude_of_projection_origin"],
 central_longitude=proj_dict["longitude_of_central_meridian"],
  standard_parallels=proj_dict["standard_parallel"], 
  globe=ccrs.Globe(semimajor_axis=proj_dict["earth_radius"], 
  semiminor_axis=proj_dict["earth_radius"]))
plot_proj